In [13]:
import base64
import os
import time
from pdf2image import convert_from_path
from PIL import Image
import io
from dotenv import load_dotenv
import numpy as np
import pandas as pd
import re
import fitz  # PyMuPDF
import json
import json_repair
from json_repair import repair_json
# 加载环境变量
load_dotenv()

import concurrent.futures
from threading import Lock

In [14]:
import os
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import List
from enum import Enum

In [15]:
# 1. 初始化客户端
client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key="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",
)

MODEL = 'openai/gpt-5-chat'
MAX_TOKENS = 8192
WAIT_TIME = 0.8 # Wait time between each request.

In [16]:
# 创建result文件夹
output_dir = "/Users/caoyushi/Downloads/性别与教材改革/2026.2.20/test"
os.makedirs(output_dir, exist_ok=True)  # 自动创建文件夹（如果不存在） 

In [17]:
# 2. 
# --- 核心技巧：在这里定义你的职业代码表 ---
# 这样模型在生成时，底层会被限制只能输出这些值，不会瞎编。
# 1. 先定义枚举类（用来替代 Literal）
# 这样做能完美生成 JSON Schema，不会报错
# 1. 定义枚举类
class ProfessionCode(str, Enum):
    TEACHER_01 = "EDU_01"   
    DOCTOR_02 = "MED_02"    
    UNKNOWN = "UNKNOWN"     

class GenderEnum(str, Enum):
    MALE = "男"
    FEMALE = "女"
    UNKNOWN = "未知"

class SourceTypeEnum(str, Enum):
    ILLUSTRATION = "插图"
    MIXED = "插图和文本"

class ScenarioEnum(str, Enum):
    HOME = "家庭场景"
    WORK = "工作场景"
    UNKNOWN = "未知"

# 2. 定义角色结构
class Character(BaseModel):
    identifier: str = Field(description="人物名称，若无名称则输出'场景+角色'的简短描述")
    
    profession_desc: str = Field(description="根据视觉判断的职业自然语言描述，如'医生'")
    
    # 只有这样，Pydantic 才能生成带有 enum 限制的 JSON Schema，强制大模型只能输出这三个值之一。
    profession_code: ProfessionCode = Field(
        description="请根据职业描述，填写对应的职业代码。如果不确定，填 'UNKNOWN'。")
    
    # 3. 这里引用上面定义的枚举类
    gender: GenderEnum = Field(description="根据视觉判断的性别")
    source_type: SourceTypeEnum = Field(description="判断依据的类型")
    scenario: ScenarioEnum = Field(description="场景类型")
    page: int = Field(default=0, exclude=True)

# 3. 定义最终输出容器
class AnalysisResult(BaseModel):
    persons: List[Character]

In [18]:
instructions = """
你是一位教材图像分析专家。请分析图片，提取所有图片中出现的“人类角色”的信息。

核心识别规则：
1. **对象限制**：仅识别图片中出现的人类。严禁输出动物、植物、神话生物或非生物拟人化角色。
2. **计数原则**：同一角色如果在图中多次出现（例如在不同分镜中），必须作为多条独立记录输出，不可合并。
3. **职业判断**：
   - 成年人：依据服饰（如白大褂、警服）和动作道具推断职业。
   - 未成年人：统一归类为“学生”或“儿童”。
   - 若无法判断，描述为“未知”。
4. **标识符**：优先提取文字姓名；若无姓名，请生成“颜色+特征+角色”的简短描述（例如：“戴眼镜的穿红衣男子”）。

请根据图片内容，准确填充预定义的字段。
"""


In [19]:
def pdf_to_images(pdf_path, dpi=200, output_folder="temp_images"):
    """
    将PDF转换为高质量的图像序列
    返回：图像路径列表和总页数
    """
    # 创建输出目录
    os.makedirs(output_folder, exist_ok=True)
    
    # 转换PDF
    images = convert_from_path(
        pdf_path, 
        dpi=dpi, 
        output_folder=output_folder,
        fmt='jpeg',
        thread_count=4,
        poppler_path=None  # 自动查找或指定路径
    )
    
    # 保存图像
    image_paths = []
    for i, image in enumerate(images):
        img_path = os.path.join(output_folder, f"page_{i+1:03d}.jpg")
        image.save(img_path, "JPEG", quality=95)
        image_paths.append(img_path)
        print(f"已转换第 {i+1} 页 -> {img_path}")
    
    return image_paths, len(images)


In [20]:
def analyze_image(image_path, page_num=None):
    """
    分析单张图像
    """
    img = Image.open(image_path)
    img_bytes = io.BytesIO()
    img.save(img_bytes, format='JPEG', quality=85)
    img_bytes = img_bytes.getvalue()
    base64_image = base64.b64encode(img_bytes).decode('utf-8')
    
    page_info = f"当前页码: {page_num}" if page_num else ""
    try:
        print(f"正在分析第 {page_num} 页..." if page_num else "正在分析图像...")
        completion = client.beta.chat.completions.parse(
            model=MODEL,  # 建议使用支持 Structured Outputs 的最新模型
            messages=[
                {"role": "system", "content": f"{instructions}\n\n{page_info}"},
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": "请分析这张图片。"},
                        {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}},
                    ],
                }
            ],
            response_format=AnalysisResult,  # 这里传入上面定义的 Pydantic 类
        )
    # 5. 获取并处理结果
        json_response = completion.choices[0].message.parsed
        if json_response and json_response.persons and page_num is not None:
                for person in json_response.persons:
                    person.page = page_num
    # 注意：这里不需要 json.loads，SDK 已经帮你转换好了
        print(f"分析结果: {completion.choices[0].message.parsed}")
        return json_response

    except Exception as e:
        print(f"调用出错: {e}")
        return None
        

In [21]:
import concurrent.futures
from threading import Lock
import threading

def process_single_image(args):
    """
    处理单张图像的函数，用于并行执行
    """
    i, img_path = args
    page_num = i + 1
    result = analyze_image(img_path, page_num)
    
    # 为每个人物添加页码信息
    if result and result.persons:
        for person in result.persons:
            person.page = page_num
    
    time.sleep(WAIT_TIME)  # 控制API请求频率
    return result

def process_all_images_parallel(image_paths, max_workers=5):
    """
    并行处理所有图像页面
    """
    combined_data = []
    lock = Lock()  # 用于保护共享资源
    
    # 使用ThreadPoolExecutor实现并行处理
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        # 提交所有任务
        future_to_args = {
            executor.submit(process_single_image, (i, img_path)): (i, img_path)
            for i, img_path in enumerate(image_paths)
        }
        
        # 收集结果
        for future in concurrent.futures.as_completed(future_to_args):
            try:
                result = future.result()
                if result and result.persons:
                    with lock:
                        combined_data.extend(result.persons)
            except Exception as e:
                print(f"处理页面时出错: {e}")
    
    # 按页码排序以保持结果顺序一致
    combined_data.sort(key=lambda x: x.page)
    return combined_data

In [22]:
def save_results(json_data, output_file):
    """保存结果到文件"""
    
    # 保存原始JSON
    with open(f"{output_file}.json", "w", encoding="utf-8") as f:
        json.dump(json_data, f, ensure_ascii=False, indent=2)
    
    # 保存Excel（通过DataFrame转换）
    df = pd.DataFrame(json_data)
    df.to_excel(f"{output_file}.xlsx", index=False)

In [24]:
# 主执行流程
if __name__ == "__main__":
    PDF_PATH = "/Users/caoyushi/Downloads/性别与教材改革/textbook/人教版_一年级下册(2025春版)_道德与法治电子课本_试验.pdf"
    #PDF_PATH = "/Users/caoyushi/Downloads/性别与教材改革/道法/人教版_1.2+品德与生活+一年级下.pdf"

    if not os.path.exists(PDF_PATH):
        print(f"错误：文件 {PDF_PATH} 不存在")
        parent_dir = os.path.dirname(PDF_PATH)
        if os.path.exists(parent_dir):
            print(f"目录内容: {os.listdir(parent_dir)}")
    else:
        print(f"开始处理: {PDF_PATH}")
        
        # 步骤1: 提取PDF文件名（不含路径和扩展名）
        pdf_filename = os.path.basename(PDF_PATH)
        if '.' in pdf_filename:
            pdf_name = pdf_filename.rsplit('.', 1)[0]  # 移除文件扩展名
        else:
            pdf_name = pdf_filename
        
        # 步骤2: 将PDF转换为图像
        print("转换PDF为图像序列...")
        image_paths, total_pages = pdf_to_images(PDF_PATH, dpi=150)
        print(f"转换完成，共 {total_pages} 页")
        
        # 步骤3: 分析所有图像页面
        print("\n开始分析教材内容...")
        combined_data= process_all_images_parallel(image_paths)
        json_data = [person.model_dump() for person in combined_data]  # 转换为json格式
        
        
        # 步骤5: 保存结果文件（修改路径指向result文件夹）
        base_output_name = f"{pdf_name}_人物识别结果_test"
        
        # 所有文件路径添加output_dir前缀
        json_file = os.path.join(output_dir, f"{base_output_name}.json")
        excel_file = os.path.join(output_dir, f"{base_output_name}.xlsx")
        
        # 保存原始JSON数据
        with open(json_file, "w", encoding="utf-8") as f:
            json.dump(json_data, f, ensure_ascii=False, indent=2)
        print(f"\n原始JSON数据已保存到 {json_file}")
        
        # 保存Excel文件
        df = pd.DataFrame(json_data)
        df.to_excel(excel_file, index=False)
        print(f"Excel文件已保存到 {excel_file}")
        
        # 清理临时图像文件（保持不变）
        import shutil
        shutil.rmtree("temp_images")
        print("已清理临时图像文件")

开始处理: /Users/caoyushi/Downloads/性别与教材改革/textbook/人教版_一年级下册(2025春版)_道德与法治电子课本_试验.pdf
转换PDF为图像序列...
已转换第 1 页 -> temp_images/page_001.jpg
已转换第 2 页 -> temp_images/page_002.jpg
已转换第 3 页 -> temp_images/page_003.jpg
已转换第 4 页 -> temp_images/page_004.jpg
已转换第 5 页 -> temp_images/page_005.jpg
已转换第 6 页 -> temp_images/page_006.jpg
已转换第 7 页 -> temp_images/page_007.jpg
已转换第 8 页 -> temp_images/page_008.jpg
已转换第 9 页 -> temp_images/page_009.jpg
已转换第 10 页 -> temp_images/page_010.jpg
转换完成，共 10 页

开始分析教材内容...
正在分析第 2 页...
正在分析第 4 页...
正在分析第 5 页...
正在分析第 1 页...
正在分析第 3 页...
分析结果: persons=[]
分析结果: persons=[]
分析结果: persons=[]
正在分析第 6 页...
正在分析第 7 页...
正在分析第 8 页...
分析结果: persons=[Character(identifier='穿粉色衣服扎粉色头绳的女孩', profession_desc='学生', profession_code=<ProfessionCode.TEACHER_01: 'EDU_01'>, gender=<GenderEnum.FEMALE: '女'>, source_type=<SourceTypeEnum.ILLUSTRATION: '插图'>, scenario=<ScenarioEnum.HOME: '家庭场景'>, page=1), Character(identifier='穿绿色上衣的男孩', profession_desc='学生', profession_code=<ProfessionCode.TEACH